In [ ]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

In [ ]:
os.chdir("/content/drive/MyDrive/Classroom")

(a)
Load all nine CSV files using pd.read_csv(). For files exceeding available RAM, use chunksize appropriately. Save the concatenated output as a Parquet file and report: (i) memory usage before and after downcasting, and (ii) file size of the resulting Parquet.



In [ ]:
# Load All CSV Files
payment_history = pd.read_csv("payment_history.csv")
monthly_emi_track = pd.read_csv("monthly_emi_track.csv")
loans_master = pd.read_csv("loans_master.csv")
loan_performance = pd.read_csv("loan_performance.csv")
loan_enquiry_bureau = pd.read_csv("loan_enquiry_bureau.csv")
customer_bureau = pd.read_csv("customer_bureau.csv")
credit_card_behavior = pd.read_csv("credit_card_behavior.csv")
collateral_assets = pd.read_csv("collateral_assets.csv")
branch_region_economy = pd.read_csv("branch_region_economy.csv")

In [ ]:
# Memory Usage Before Downcasting
memory_before = sum(
    df.memory_usage(deep=True).sum()
    for df in [
        loans_master,
        loan_performance,
        payment_history,
        monthly_emi_track,
        loan_enquiry_bureau,
        customer_bureau,
        credit_card_behavior,
        collateral_assets,
        branch_region_economy
    ]
)

print(f"Memory Before: {memory_before / 1024**2:.2f} MB")

Memory Before: 7524.14 MB


In [ ]:
# Downcast Numeric Columns
def reduce_memory(df):

    for col in df.select_dtypes(include=['int64']).columns:
        df[col] = pd.to_numeric(df[col], downcast='integer')

    for col in df.select_dtypes(include=['float64']).columns:
        df[col] = pd.to_numeric(df[col], downcast='float')

    return df

In [ ]:
files = [
    loans_master,
    loan_performance,
    payment_history,
    monthly_emi_track,
    loan_enquiry_bureau,
    customer_bureau,
    credit_card_behavior,
    collateral_assets,
    branch_region_economy
]

for df in files:
    reduce_memory(df)

In [ ]:
# Memory After Downcasting
memory_after = sum(
    df.memory_usage(deep=True).sum()
    for df in files
)

print(f"Memory After: {memory_after / 1024**2:.2f} MB")

Memory After: 6408.34 MB


(b)
Join all nine tables on loan_id using sequential left merges. After every individual join, assert that the running row count equals 2,000,000. Report the number of orphan records found, if any, and explain what orphan records indicate about data integrity.



In [ ]:
# Verify Row Counts Before Merging
# Check for Duplicate loan_id
files = {
    "loans_master": loans_master,
    "loan_performance": loan_performance,
    "payment_history": payment_history,
    "monthly_emi_track": monthly_emi_track,
    "loan_enquiry_bureau": loan_enquiry_bureau,
    "customer_bureau": customer_bureau,
    "credit_card_behavior": credit_card_behavior,
    "collateral_assets": collateral_assets,
    "branch_region_economy": branch_region_economy
}

for name, df in files.items():

    total_rows = len(df)
    unique_loans = df['loan_id'].nunique()
    duplicate_count = df['loan_id'].duplicated().sum()

    print(f"\n{name}")
    print(f"Total Rows      : {total_rows:,}")
    print(f"Unique loan_ids : {unique_loans:,}")
    print(f"Duplicate IDs   : {duplicate_count:,}")


loans_master
Total Rows      : 2,000,000
Unique loan_ids : 2,000,000
Duplicate IDs   : 0

loan_performance
Total Rows      : 2,000,000
Unique loan_ids : 2,000,000
Duplicate IDs   : 0

payment_history
Total Rows      : 2,000,000
Unique loan_ids : 2,000,000
Duplicate IDs   : 0

monthly_emi_track
Total Rows      : 2,000,000
Unique loan_ids : 2,000,000
Duplicate IDs   : 0

loan_enquiry_bureau
Total Rows      : 2,000,000
Unique loan_ids : 2,000,000
Duplicate IDs   : 0

customer_bureau
Total Rows      : 2,000,000
Unique loan_ids : 2,000,000
Duplicate IDs   : 0

credit_card_behavior
Total Rows      : 2,000,000
Unique loan_ids : 2,000,000
Duplicate IDs   : 0

collateral_assets
Total Rows      : 2,000,000
Unique loan_ids : 2,000,000
Duplicate IDs   : 0

branch_region_economy
Total Rows      : 2,000,000
Unique loan_ids : 2,000,000
Duplicate IDs   : 0


In [ ]:
# Orphan Record Detection
tables = {
    "loan_performance": loan_performance,
    "payment_history": payment_history,
    "monthly_emi_track": monthly_emi_track,
    "loan_enquiry_bureau": loan_enquiry_bureau,
    "customer_bureau": customer_bureau,
    "credit_card_behavior": credit_card_behavior,
    "collateral_assets": collateral_assets,
    "branch_region_economy": branch_region_economy
}

for name, df in tables.items():

    orphan_count = (
        ~df['loan_id'].isin(loans_master['loan_id'])
    ).sum()

    print(f"{name}: {orphan_count}")

loan_performance: 0
payment_history: 0
monthly_emi_track: 0
loan_enquiry_bureau: 0
customer_bureau: 0
credit_card_behavior: 0
collateral_assets: 0
branch_region_economy: 0


(c)
The dataset contains eight deliberately injected data quality issues across multiple columns. Identify all eight, create a binary dirty_flag column to mark affected rows, and for each issue state: the column affected, the approximate count of dirty records, why the value is invalid, and the imputation strategy you applied.



In [ ]:
master_df = loans_master.copy()

In [ ]:
merge_tables = [
    loan_performance,
    payment_history,
    monthly_emi_track,
    loan_enquiry_bureau,
    customer_bureau,
    credit_card_behavior,
    collateral_assets,
    branch_region_economy
]

In [ ]:
for i, table in enumerate(merge_tables, start=1):

    master_df = master_df.merge(
        table,
        on='loan_id',
        how='left'
    )

    print(
        f"After Join {i}: {master_df.shape}"
    )

    assert len(master_df) == 2000000

After Join 1: (2000000, 38)
After Join 2: (2000000, 55)
After Join 3: (2000000, 77)
After Join 4: (2000000, 100)
After Join 5: (2000000, 129)
After Join 6: (2000000, 145)
After Join 7: (2000000, 164)
After Join 8: (2000000, 182)


In [ ]:
master_df.isnull().sum().sort_values(ascending=False).head(20)

,0
vehicle_type,1879789
property_type,1679672
property_city_tier,1679672
mths_since_last_record,1598332
ltv_ratio_pct,1472196
valuation_agency,1370048
collateral_type,1258896
charge_type,1258896
mths_since_last_delinq,1098640
primary_card_type,560528


In [ ]:
missing_pct = (
    master_df.isnull().sum()
    / len(master_df)
    * 100
).sort_values(ascending=False)

missing_pct.head(20)

,0
vehicle_type,93.98945
property_type,83.98360
property_city_tier,83.98360
mths_since_last_record,79.91660
ltv_ratio_pct,73.60980
valuation_agency,68.50240
collateral_type,62.94480
charge_type,62.94480
mths_since_last_delinq,54.93200
primary_card_type,28.02640


(d)
Classify the missing-value pattern for each high-missing column (mths_since_last_delinq, mort_acc, emp_length_years, il_util_pct) as MCAR, MAR, or MNAR. Justify each classification using either Little's MCAR test output or domain reasoning. Apply the correct imputation strategy and verify with .isnull().sum() before and after.



In [ ]:
# Missing Value Classification
master_df['mths_since_last_delinq'] = (
    master_df['mths_since_last_delinq']
    .fillna(999)
)

In [ ]:
master_df['mort_acc'] = (
    master_df.groupby('home_ownership')
    ['mort_acc']
    .transform(
        lambda x: x.fillna(x.median())
    )
)

In [ ]:
master_df['emp_length_years'] = (
    master_df.groupby('emp_title')
    ['emp_length_years']
    .transform(
        lambda x: x.fillna(x.median())
    )
)

In [ ]:
master_df['il_util_pct'] = (
    master_df['il_util_pct']
    .fillna(0)
)

In [ ]:
# Verification
master_df[
[
'mths_since_last_delinq',
'mort_acc',
'emp_length_years',
'il_util_pct'
]
].isnull().sum()

,0
mths_since_last_delinq,0
mort_acc,0
emp_length_years,0
il_util_pct,0


In [ ]:
cols = [
    'age',
    'annual_inc_inr',
    'loan_amnt_inr',
    'funded_amnt_inr',
    'int_rate_pct',
    'dti_pct',
    'cibil_score',
    'cc_utilization_pct',
    'collateral_value_inr',
    'revol_util_pct',
    'bc_util_pct'
]

master_df[cols].describe().T

,count,mean,std,min,25%,50%,75%,max
age,2000000.0,38.186577,9.608274e+00,21.00,31.000000,38.000000,45.000000,7.500000e+01
annual_inc_inr,1959934.0,545640.807532,6.059026e+05,100000.00,197327.000000,362293.000000,664705.500000,2.000000e+07
loan_amnt_inr,2000000.0,150338.859375,1.335037e+05,50000.00,64541.000000,109172.000000,184666.000000,5.000000e+06
funded_amnt_inr,2000000.0,141323.218750,1.254991e+05,44000.00,60587.000000,102546.000000,173600.000000,4.810737e+06
int_rate_pct,2000000.0,13.574755,4.373115e+00,7.00,10.120000,12.860000,16.490000,2.800000e+01
dti_pct,2000000.0,17.151524,9.575985e+00,0.01,9.680000,15.870000,23.370001,5.747000e+01
cibil_score,2000000.0,679.882791,8.462665e+01,300.00,623.000000,680.000000,737.000000,9.000000e+02
cc_utilization_pct,2000000.0,23.997835,2.127497e+01,0.00,0.000000,22.100000,39.400002,9.860000e+01
collateral_value_inr,2000000.0,454128.908525,1.074111e+06,0.00,0.000000,0.000000,288978.500000,2.557225e+07
revol_util_pct,1859903.0,39.996552,2.000164e+01,0.00,24.299999,38.599998,54.400002,9.980000e+01


(e)
Apply winsorisation at the 1st and 99th percentile to the six most skewed numeric columns. Present a before-and-after comparison of mean, standard deviation, and max for each column in a summary table.


In [ ]:
# Six Most Skewed Variables
numeric_cols = master_df.select_dtypes(include=np.number).columns

skewness = (
    master_df[numeric_cols]
    .skew()
    .abs()
    .sort_values(ascending=False)
)

top6_cols = skewness.head(6)

print(top6_cols)

npa_flag                   365.144538
collections_12mths_fee     139.035412
collection_recovery_fee    125.765963
recoveries_inr              94.232480
emi_advance_paid_inr        59.171673
expected_loss_inr           27.014741
dtype: float64


In [ ]:
# winsorization:
summary = []

for col in top6_cols.index:

    mean_before = master_df[col].mean()
    std_before = master_df[col].std()
    max_before = master_df[col].max()

    p1 = master_df[col].quantile(0.01)
    p99 = master_df[col].quantile(0.99)

    master_df[col] = master_df[col].clip(
        lower=p1,
        upper=p99
    )

    mean_after = master_df[col].mean()
    std_after = master_df[col].std()
    max_after = master_df[col].max()

    summary.append([
        col,
        mean_before,
        mean_after,
        std_before,
        std_after,
        max_before,
        max_after
    ])

In [ ]:
winsor_summary = pd.DataFrame(
    summary,
    columns=[
        'Column',
        'Mean Before',
        'Mean After',
        'Std Before',
        'Std After',
        'Max Before',
        'Max After'
    ]
)

winsor_summary.round(2)

,Column,Mean Before,Mean After,Std Before,Std After,Max Before,Max After
0,npa_flag,0.00,0.00,0.00,0.00,1.00,0.00
1,collections_12mths_fee,32.37,11.83,560.07,77.30,312618.52,657.22
2,collection_recovery_fee,130.71,55.19,2061.44,348.53,1035421.73,2911.07
3,recoveries_inr,52.75,22.14,818.40,140.02,313742.97,1172.06
4,emi_advance_paid_inr,2204.31,1708.03,13599.90,5752.16,5329334.00,40329.12
5,expected_loss_inr,139.24,84.35,1285.91,500.60,189646.84,3990.42


In [ ]:
master_df.to_parquet(
    "credit_risk_cleaned.parquet",
    index=False
)